# K-Nearest Neighbors (K-NN): estudio detallado y propuesta de aplicación
Cuaderno autocontenido (no descarga nada; usa datasets incluidos en scikit-learn). Ejecute con *Entorno de ejecución → Ejecutar todo*.

**Contenido**
1. Documentación del algoritmo (idea, pasos, distancias, k, pesos, escalado, dimensionalidad, complejidad)
2. Implementación desde cero, documentada y verificada contra scikit-learn
3. Experimentos que ilustran cada punto delicado
4. Propuesta de aplicación: **recomendador explicable basado en K-NN**, con los cambios al algoritmo y una muestra mínima

> La sección 4 es un **planteamiento** y una muestra mínima, no una aplicación terminada.

## 1. Documentación del algoritmo

### 1.1 Idea
K-NN es un método **basado en instancias** ("aprendizaje perezoso"): no ajusta parámetros durante el entrenamiento, solo **guarda los datos**. Para predecir un punto nuevo busca los *k* ejemplos más parecidos (sus vecinos) y deduce la respuesta a partir de ellos. Supone que *puntos cercanos en el espacio de features tienden a tener la misma etiqueta* (hipótesis de suavidad local).

### 1.2 Algoritmo paso a paso
```
Entrada: datos de entrenamiento (X, y), k, métrica d(·,·), esquema de pesos, punto nuevo x
1. Calcular d(x, xi) para cada ejemplo xi de entrenamiento.
2. Seleccionar los k ejemplos con menor distancia  -> vecindario N_k(x).
3. Clasificación: contar votos por clase (uniformes, o ponderados por 1/d).  Predecir la clase con más votos.
   Regresión:     predecir el promedio (simple o ponderado) de los y de los vecinos.
4. (Opcional) Probabilidad de clase c = (suma de pesos de vecinos de clase c) / (suma de todos los pesos).
```

### 1.3 Distancias
| Métrica | Fórmula | Cuándo usarla |
|---|---|---|
| Euclidiana (L2) | $\sqrt{\sum_j (a_j-b_j)^2}$ | Valor por defecto, features continuas y escaladas |
| Manhattan (L1) | $\sum_j \lvert a_j-b_j\rvert$ | Más robusta a valores atípicos; útil en alta dimensión |
| Minkowski (Lp) | $\left(\sum_j \lvert a_j-b_j\rvert^p\right)^{1/p}$ | Generaliza L1 ($p=1$) y L2 ($p=2$) |
| Chebyshev (L∞) | $\max_j \lvert a_j-b_j\rvert$ | Importa la mayor diferencia individual |
| Coseno | $1-\dfrac{a\cdot b}{\lVert a\rVert\lVert b\rVert}$ | Texto, embeddings: importa la dirección, no la magnitud |
| Hamming | fracción de posiciones distintas | Variables categóricas / binarias |
| Gower | promedio de distancias por feature normalizadas a [0,1] | **Datos mezclados** (numéricos + categóricos); se usa en la sección 4 |

### 1.4 Elección de k (sesgo–varianza)
- **k pequeño** (p. ej. 1): frontera muy irregular, sensible al ruido → *varianza alta, sobreajuste*. Con k=1 el error de entrenamiento es 0.
- **k grande**: frontera suave, promedia mucho → *sesgo alto, subajuste*. Con k = n se predice siempre la clase mayoritaria.
- Se elige con **validación cruzada**. En clasificación binaria conviene un k **impar** para evitar empates. Regla de partida: $k\approx\sqrt{n}$.

### 1.5 Ponderación de vecinos
Con pesos $w_i = 1/d_i$ los vecinos cercanos pesan más. Reduce la sensibilidad a la elección de k y ayuda cuando el vecindario mezcla puntos lejanos. Si algún $d_i=0$ (punto idéntico) se usa solo ese vecino.

### 1.6 Escalado de features
La distancia mezcla las unidades de todas las variables: una feature en miles domina a una en unidades. **Estandarizar** (media 0, desviación 1) o normalizar a [0,1] es casi obligatorio. Calcule los parámetros del escalado **solo con entrenamiento** (evita fuga de datos).

### 1.7 Maldición de la dimensionalidad
En muchas dimensiones las distancias se concentran: el vecino más cercano y el más lejano quedan casi a la misma distancia y "cercano" pierde sentido. Además, cada feature irrelevante añade ruido a la distancia. Remedios: seleccionar features, reducir dimensión (PCA), ponderar features, usar métricas L1/coseno.

### 1.8 Complejidad
| Método | Construcción | Consulta | Notas |
|---|---|---|---|
| Fuerza bruta | O(1) | O(n·d) | Exacto; sencillo; bueno para n pequeño o d grande |
| KD-tree | O(n log n) | ≈ O(log n) en d baja | Se degrada si d ≳ 20 |
| Ball-tree | O(n log n) | mejor que KD en d media | Admite más métricas |
| Búsqueda aproximada (ANN: LSH, HNSW, FAISS, Annoy) | variable | sublineal | Para millones de puntos; se sacrifica algo de exactitud |

La memoria es O(n·d): hay que **conservar todo el entrenamiento**, y cada predicción es costosa (lo contrario de la mayoría de modelos).

### 1.9 Ventajas, límites y buenas prácticas
**Ventajas:** simple, sin fase de entrenamiento, no paramétrico (frontera arbitraria), multiclase natural, **explicable** (se pueden mostrar los vecinos).
**Límites:** costoso al predecir, sensible a escala/irrelevantes/dimensión, mal con clases muy desbalanceadas (domina la mayoritaria), no extrapola en regresión, necesita buena métrica.
**Checklist:** limpiar/imputar → escalar → seleccionar features → elegir métrica → validar k con CV → considerar pesos → evaluar con métricas adecuadas al desbalance → usar índice (árbol/ANN) si n crece.

In [ ]:
import time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
warnings.filterwarnings("ignore")
SEED = 0; np.random.seed(SEED)
plt.rcParams["figure.dpi"] = 90

## 2. Implementación desde cero (documentada)
Soporta clasificación y regresión, seis métricas, pesos uniformes o por inverso de la distancia, y probabilidades.

In [ ]:
class KNN:
    """K-Nearest Neighbors desde cero (numpy).

    Parametros
    ----------
    k : int            Número de vecinos.
    metric : str       'euclidean' | 'manhattan' | 'chebyshev' | 'minkowski' | 'cosine' | 'hamming'.
    p : float          Exponente para 'minkowski'.
    weights : str      'uniform' (voto simple) o 'distance' (peso 1/d).
    task : str         'classification' o 'regression'.

    Notas
    -----
    * fit() solo almacena los datos (aprendizaje perezoso).
    * Desempate en clasificación: gana la clase de menor etiqueta (igual que scikit-learn).
    """
    def __init__(self, k=5, metric="euclidean", p=3, weights="uniform", task="classification"):
        assert metric in ("euclidean", "manhattan", "chebyshev", "minkowski", "cosine", "hamming")
        assert weights in ("uniform", "distance") and task in ("classification", "regression")
        self.k, self.metric, self.p, self.weights, self.task = k, metric, p, weights, task

    def fit(self, X, y):
        self.X_ = np.asarray(X, float); self.y_ = np.asarray(y)
        assert 1 <= self.k <= len(self.X_), "k debe estar entre 1 y n"
        if self.task == "classification":
            self.clases_, self.yi_ = np.unique(self.y_, return_inverse=True)
        return self

    def _dist(self, A, B):
        """Matriz de distancias (len(A) x len(B))."""
        m = self.metric
        if m == "euclidean":                       # ||a-b||² = ||a||² + ||b||² - 2a·b
            d2 = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
            return np.sqrt(np.maximum(d2, 0))
        if m == "cosine":
            na, nb = np.linalg.norm(A, axis=1)[:, None], np.linalg.norm(B, axis=1)[None, :]
            return 1 - (A @ B.T) / np.maximum(na * nb, 1e-12)
        if m == "hamming":
            return (A[:, None, :] != B[None, :, :]).mean(2)
        diff = np.abs(A[:, None, :] - B[None, :, :])
        if m == "manhattan": return diff.sum(2)
        if m == "chebyshev": return diff.max(2)
        return (diff ** self.p).sum(2) ** (1 / self.p)   # minkowski

    def kneighbors(self, X, chunk=500):
        """Devuelve (distancias, índices) de los k vecinos, ordenados de menor a mayor distancia."""
        X = np.asarray(X, float); D_out, I_out = [], []
        for s in range(0, len(X), chunk):            # por bloques: controla la memoria
            D = self._dist(X[s:s + chunk], self.X_)
            idx = np.argpartition(D, self.k - 1, axis=1)[:, :self.k]   # k menores sin ordenar todo (O(n))
            d = np.take_along_axis(D, idx, 1)
            o = np.argsort(d, axis=1)
            D_out.append(np.take_along_axis(d, o, 1)); I_out.append(np.take_along_axis(idx, o, 1))
        return np.vstack(D_out), np.vstack(I_out)

    def _pesos(self, d):
        if self.weights == "uniform": return np.ones_like(d)
        cero = d == 0
        w = 1.0 / np.where(cero, 1.0, d)
        filas = cero.any(1)
        w[filas] = cero[filas].astype(float)         # si hay puntos idénticos, solo ellos votan
        return w

    def predict_proba(self, X):
        assert self.task == "classification"
        d, idx = self.kneighbors(X); w = self._pesos(d)
        votos = (self.yi_[idx][..., None] == np.arange(len(self.clases_))) * w[..., None]
        return votos.sum(1) / w.sum(1, keepdims=True)

    def predict(self, X):
        if self.task == "classification":
            return self.clases_[self.predict_proba(X).argmax(1)]
        d, idx = self.kneighbors(X); w = self._pesos(d)
        return (w * self.y_[idx]).sum(1) / w.sum(1)

    def score(self, X, y):
        p = self.predict(X)
        return (p == np.asarray(y)).mean() if self.task == "classification" else 1 - ((y - p)**2).sum() / ((y - np.mean(y))**2).sum()
print("Clase KNN definida.")

### 2.1 Verificación contra scikit-learn
Si la implementación es correcta, debe coincidir con `KNeighborsClassifier` (datos continuos y escalados, sin empates de distancia).

In [ ]:
Xw, yw = load_wine(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(Xw, yw, test_size=0.3, stratify=yw, random_state=SEED)
esc = StandardScaler().fit(Xtr); Xtr_s, Xte_s = esc.transform(Xtr), esc.transform(Xte)
filas = []
for metric in ["euclidean", "manhattan", "chebyshev"]:
    for w in ["uniform", "distance"]:
        mio = KNN(k=5, metric=metric, weights=w).fit(Xtr_s, ytr)
        sk = KNeighborsClassifier(n_neighbors=5, metric=metric, weights=w).fit(Xtr_s, ytr)
        coincide = (mio.predict(Xte_s) == sk.predict(Xte_s)).mean()
        dif_proba = np.abs(mio.predict_proba(Xte_s) - sk.predict_proba(Xte_s)).max()
        filas.append({"metrica": metric, "pesos": w, "acc propia": round(mio.score(Xte_s, yte), 3),
                      "acc sklearn": round(sk.score(Xte_s, yte), 3), "% predicciones iguales": 100 * coincide, "dif. máx. proba": dif_proba})
tab = pd.DataFrame(filas); display(tab)
assert (tab["% predicciones iguales"] == 100).all() and (tab["dif. máx. proba"] < 1e-9).all()
print("OK: la implementación propia coincide con scikit-learn.")

## 3. Experimentos
### 3.1 Efecto de k en la frontera de decisión (iris, 2 features)

In [ ]:
iris = load_iris(); X2, y2 = iris.data[:, 2:4], iris.target
Xa, Xb, ya, yb = train_test_split(X2, y2, test_size=0.3, stratify=y2, random_state=SEED)
gx, gy = np.meshgrid(np.linspace(0.5, 7.2, 150), np.linspace(-0.1, 2.8, 150))
malla = np.c_[gx.ravel(), gy.ravel()]
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, k in zip(axs, [1, 15, 75]):
    m = KNN(k=k).fit(Xa, ya)
    ax.contourf(gx, gy, m.predict(malla).reshape(gx.shape), alpha=0.3, cmap="viridis")
    ax.scatter(Xa[:, 0], Xa[:, 1], c=ya, edgecolor="k", cmap="viridis")
    ax.set_title(f"k={k} | acc train={m.score(Xa, ya):.2f} | acc test={m.score(Xb, yb):.2f}")
    ax.set_xlabel("largo del pétalo"); ax.set_ylabel("ancho del pétalo")
plt.tight_layout(); plt.show()

**Lectura:** con k=1 la frontera sigue cada punto (train=1.00, sobreajuste); con k muy grande se vuelve demasiado simple y pierde exactitud.

### 3.2 Elección de k por validación cruzada, y efecto del escalado (wine)

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
ks = range(1, 41)
esc_cv  = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xw, yw, cv=cv).mean() for k in ks]
raw_cv  = [cross_val_score(KNeighborsClassifier(k), Xw, yw, cv=cv).mean() for k in ks]
tr_acc  = [make_pipeline(StandardScaler(), KNeighborsClassifier(k)).fit(Xw, yw).score(Xw, yw) for k in ks]
plt.plot(ks, esc_cv, label="CV con escalado"); plt.plot(ks, raw_cv, label="CV sin escalar"); plt.plot(ks, tr_acc, "--", label="entrenamiento (escalado)")
plt.xlabel("k"); plt.ylabel("accuracy"); plt.legend(); plt.title("Accuracy vs k"); plt.show()
k_best = list(ks)[int(np.argmax(esc_cv))]
print(f"Mejor k (escalado): {k_best} -> {max(esc_cv):.3f} | mejor sin escalar: {max(raw_cv):.3f}")

### 3.3 Comparación de métricas (wine, escalado, k=5)

In [ ]:
res = {m: cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5, metric=m)), Xw, yw, cv=cv).mean()
       for m in ["euclidean", "manhattan", "chebyshev", "cosine"]}
display(pd.Series(res, name="accuracy CV").round(3).to_frame())

### 3.4 Maldición de la dimensionalidad y features irrelevantes

In [ ]:
rng = np.random.default_rng(SEED)
dims = [1, 2, 5, 10, 20, 50, 100, 500, 1000]
contraste = []
for d in dims:
    P = rng.random((1000, d)); r = []
    for _ in range(20):
        q = rng.random(d); dist = np.linalg.norm(P - q, axis=1); r.append((dist.max() - dist.min()) / dist.min())
    contraste.append(np.mean(r))
ruido = [0, 5, 10, 20, 50, 100]; acc_ruido = []
for m in ruido:
    Xn = np.c_[Xw, rng.normal(size=(len(Xw), m))] if m else Xw
    acc_ruido.append(cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(5)), Xn, yw, cv=cv).mean())
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].semilogy(dims, contraste, "o-"); ax[0].set_xscale("log"); ax[0].set_xlabel("dimensiones")
ax[0].set_title("Contraste (dmax-dmin)/dmin de distancias aleatorias")
ax[1].plot(ruido, acc_ruido, "o-", color="#e45756"); ax[1].set_xlabel("features irrelevantes añadidas a wine (13 reales)")
ax[1].set_ylabel("accuracy CV"); ax[1].set_title("K-NN se degrada con ruido")
plt.tight_layout(); plt.show()

**Lectura:** el contraste cae casi a cero al crecer la dimensión (todos los puntos parecen equidistantes) y el accuracy baja al añadir features sin información. Esto motiva **ponderar/seleccionar features** en la aplicación (sección 4).

### 3.5 Costo de predecir: fuerza bruta vs. árboles

In [ ]:
filas = []
for n in [2000, 20000]:
    for d in [3, 40]:
        P = rng.random((n, d)); lab = (P[:, 0] > 0.5).astype(int); Q = rng.random((300, d))
        fila = {"n": n, "d": d}
        for alg in ["brute", "kd_tree", "ball_tree"]:
            m = KNeighborsClassifier(5, algorithm=alg).fit(P, lab)
            t0 = time.perf_counter(); m.predict(Q); fila[alg + " (ms)"] = round(1000 * (time.perf_counter() - t0), 1)
        filas.append(fila)
display(pd.DataFrame(filas))

**Lectura:** en dimensión baja (d=3) los árboles ganan al crecer n; con d=40 la fuerza bruta es mucho más rápida que los árboles. Con millones de puntos se usa búsqueda aproximada (ANN).

## 4. De K-NN a una aplicación: recomendador explicable de cursos

### 4.1 Planteamiento
**Problema:** sugerir cursos relevantes a una persona a partir de lo que ya le gustó (o de una descripción de lo que busca), **y decir por qué**.
**Por qué K-NN:** recomendar es "encontrar ítems parecidos"; no necesita entrenamiento, funciona con poco historial, se actualiza al agregar ítems y es explicable.

### 4.2 Cambios al algoritmo estándar
| Limitación del K-NN estándar | Cambio propuesto | Dónde se ve en la muestra |
|---|---|---|
| Distancia euclidiana solo sirve con features numéricas | **Distancia tipo Gower**: numéricas normalizadas por rango, nivel ordinal, **temas con Jaccard** | `_componentes()` |
| Todas las features pesan igual (sensible a irrelevantes) | **Pesos por feature** configurables (o aprendibles después) | parámetro `pesos` |
| Se pregunta por *un* punto | **Perfil = varios ítems gustados**: distancia promedio a todos ellos | `recomendar(gustados=[...])` |
| Siempre devuelve k vecinos aunque estén lejos | **Distancia máxima**: si nadie es suficientemente parecido, no recomienda ("abstención") | parámetro `max_dist` |
| Es una caja negra para el usuario | **Explicación**: temas compartidos y rasgo más similar | columna `por_que` |
| Sin historial (arranque en frío) | **Consulta por atributos**: se describe el ítem ideal y se buscan sus vecinos | `recomendar_consulta()` |

### 4.3 Flujo propuesto
```
 Catálogo de cursos ──► Vectorización (nivel, horas, precio, rating, temas)
                                   │
 Perfil del usuario ──► ítems gustados  o  consulta por atributos
                                   │
                 Distancia mixta ponderada (Gower) a cada ítem del catálogo
                                   │
          Orden por distancia ─► filtro max_dist ─► excluir ya vistos ─► top-k
                                   │
                 Resultado + explicación (por qué se recomienda)
```

In [ ]:
class KNNRecomendador:
    """Muestra mínima: K-NN con distancia mixta ponderada, perfil multi-ítem, abstención y explicación."""
    NIV = {"basico": 0, "intermedio": 1, "avanzado": 2}
    NUM = ["horas", "precio_usd", "rating"]

    def __init__(self, pesos=None, max_dist=None):
        self.w = {"nivel": 1.0, "horas": 0.5, "precio_usd": 0.5, "rating": 0.5, "temas": 3.0, **(pesos or {})}
        self.max_dist = max_dist

    def fit(self, df):
        self.df = df.reset_index(drop=True)
        self.nivel_ = self.df.nivel.map(self.NIV).to_numpy()
        self.rango_ = (self.df[self.NUM].max() - self.df[self.NUM].min())
        self.T = self.df.temas.str.get_dummies("|").astype(bool)      # matriz multi-hot de temas
        return self

    def _item(self, i):
        r = self.df.loc[i]
        return {"nivel": r.nivel, "horas": r.horas, "precio_usd": r.precio_usd, "rating": r.rating, "temas": r.temas.split("|")}

    def _componentes(self, item):
        """Distancia (0..1) de un ítem a todo el catálogo, separada por feature."""
        c = {"nivel": np.abs(self.nivel_ - self.NIV[item["nivel"]]) / 2}
        for f in self.NUM:
            c[f] = np.clip(np.abs(self.df[f].to_numpy() - item[f]) / self.rango_[f], 0, 1)
        t = np.array([col in item["temas"] for col in self.T.columns])
        inter, union = (self.T.to_numpy() & t).sum(1), (self.T.to_numpy() | t).sum(1)
        c["temas"] = 1 - inter / np.maximum(union, 1)                    # distancia de Jaccard
        return c

    def _recomendar(self, items, excluir, n):
        comps = [self._componentes(it) for it in items]
        prom = {f: np.mean([c[f] for c in comps], axis=0) for f in self.w}      # perfil = promedio sobre ítems
        D = sum(self.w[f] * prom[f] for f in self.w) / sum(self.w.values())    # Gower ponderado
        D = pd.Series(D, index=self.df.index).drop(index=excluir)
        D = D.sort_values().head(n)
        if self.max_dist is not None: D = D[D <= self.max_dist]                # abstención
        temas_perfil = {t for it in items for t in it["temas"]}
        filas = []
        for i, d in D.items():
            comp = [t for t in self.df.loc[i, "temas"].split("|") if t in temas_perfil]
            mejor = min(self.w, key=lambda f: self.w[f] * prom[f][i])
            filas.append({"curso": self.df.loc[i, "curso"], "distancia": round(d, 3), "similitud": round(1 - d, 3),
                          "por_que": f"comparte temas: {', '.join(comp) or 'ninguno'}; rasgo más parecido: {mejor}"})
        return pd.DataFrame(filas)

    def recomendar(self, gustados, n=3):
        idx = [int(self.df.index[self.df.curso == g][0]) for g in gustados]
        return self._recomendar([self._item(i) for i in idx], idx, n)

    def recomendar_consulta(self, nivel, horas, precio_usd, rating, temas, n=3):
        item = {"nivel": nivel, "horas": horas, "precio_usd": precio_usd, "rating": rating, "temas": temas}
        return self._recomendar([item], [], n)
print("KNNRecomendador definido.")

### 4.4 Muestra mínima
Catálogo inventado de 20 cursos (solo para ilustrar).

In [ ]:
cat = pd.DataFrame([
 ("Python desde cero","basico",20,30,4.6,"python|programacion"), ("Python intermedio","intermedio",30,45,4.5,"python|programacion"),
 ("Análisis de datos con Pandas","intermedio",25,40,4.7,"python|datos"), ("Estadística para ciencia de datos","intermedio",35,50,4.4,"datos|matematicas"),
 ("Machine Learning práctico","intermedio",40,60,4.8,"python|datos|ia"), ("Redes neuronales profundas","avanzado",50,90,4.7,"python|ia|matematicas"),
 ("SQL esencial","basico",15,25,4.5,"datos|bases"), ("Bases de datos avanzadas","avanzado",40,80,4.3,"bases|datos"),
 ("HTML y CSS desde cero","basico",18,20,4.4,"web|diseno"), ("JavaScript moderno","intermedio",30,40,4.6,"web|programacion"),
 ("React para aplicaciones web","avanzado",35,70,4.5,"web|programacion"), ("Diseño UX básico","basico",12,35,4.2,"diseno|ux"),
 ("Prototipado UI en Figma","intermedio",16,45,4.5,"diseno|ux|web"), ("Visualización de datos","intermedio",22,40,4.6,"datos|diseno"),
 ("Álgebra lineal aplicada","intermedio",30,35,4.3,"matematicas|ia"), ("IA generativa para negocio","basico",10,55,4.1,"ia|negocio"),
 ("Gestión de proyectos ágiles","basico",14,30,4.0,"negocio|gestion"), ("Analítica para negocios","intermedio",24,50,4.4,"datos|negocio"),
 ("Ciberseguridad básica","basico",20,40,4.3,"seguridad|redes"), ("Fundamentos de redes","basico",18,30,4.2,"redes|seguridad")],
 columns=["curso","nivel","horas","precio_usd","rating","temas"])
rec = KNNRecomendador().fit(cat)

print("A) Perfil: le gustaron 'Python desde cero' y 'Análisis de datos con Pandas'")
display(rec.recomendar(["Python desde cero", "Análisis de datos con Pandas"], n=3))

print("B) Mismo perfil, pero priorizando precio y nivel sobre los temas (cambia el resultado)")
display(KNNRecomendador(pesos={"temas": 0.5, "precio_usd": 3.0, "nivel": 2.0}).fit(cat)
        .recomendar(["Python desde cero", "Análisis de datos con Pandas"], n=3))

print("C) Arranque en frío: describe lo que busca (sin historial)")
display(rec.recomendar_consulta("intermedio", 30, 45, 4.5, ["diseno", "web"], n=3))

print("D) Abstención: con max_dist=0.45 solo se devuelven los vecinos realmente cercanos (se pidieron 3, no se fuerzan los lejanos)")
display(KNNRecomendador(max_dist=0.45).fit(cat).recomendar(["Ciberseguridad básica"], n=3))

### 4.5 Cómo podría evolucionar (sin definir aún la implementación)
- **Datos:** catálogo con atributos (los de la muestra + embeddings de la descripción con distancia coseno) e historial de interacciones (vistos, completados, calificados).
- **Aprendizaje de pesos:** ajustar los pesos por feature a partir de las interacciones reales (p. ej. búsqueda en rejilla o regresión logística sobre pares "gustó/no gustó").
- **Escala:** cuando el catálogo crezca, reemplazar la fuerza bruta por un índice ANN (HNSW/FAISS/Annoy).
- **Diversidad:** evitar que las k recomendaciones sean casi iguales (reordenar con MMR).
- **Evaluación:** *hit-rate@k* y *precision@k* ocultando ítems que el usuario sí consumió; cobertura del catálogo, diversidad y prueba A/B con usuarios.
- **Riesgos:** arranque en frío de ítems nuevos, sesgo de popularidad, "burbuja" de recomendaciones, sensibilidad a la elección de pesos y a la escala de las features.

In [ ]:
print("RESUMEN DE LO APRENDIDO")
print(f"- La implementación propia coincide 100% con scikit-learn en {len(tab)} configuraciones.")
print(f"- Mejor k por CV en wine (escalado): {k_best}; escalar mejora el accuracy de {max(raw_cv):.3f} a {max(esc_cv):.3f}.")
print(f"- Con 100 features irrelevantes el accuracy cae de {acc_ruido[0]:.3f} a {acc_ruido[-1]:.3f}: hay que ponderar/seleccionar features.")
print(f"- El contraste de distancias pasa de {contraste[0]:.1f} (d=1) a {contraste[-1]:.3f} (d=1000): la distancia pierde poder discriminante.")
print("- La aplicación propuesta adapta K-NN con distancia mixta, pesos, perfil multi-ítem, abstención y explicaciones.")